# 04 — Forecasting models

**Phase 4 deliverable:** all 18 upstream architectures scored against `NaiveLag` on KBANK in one
command.

18 notebooks, 4 families. The upstream `deep-learning/` set is one train loop with
`{cell} × {bidirectional} × {paths} × {decoder}` — so it is implemented once and configured 18
times. Same coverage, roughly a fifth of the code, and "compare everything on KBANK" becomes a
single command instead of 18 manual runs.

In [5]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [6]:
from stock_retrofit.config import all_model_specs
from stock_retrofit.models import registered_kinds

print("model families:", ", ".join(registered_kinds()), "\n")
for spec in all_model_specs():
    print(f"  {spec.name:32s} {spec.kind:28s} <- {spec.upstream}")

model families: arima, attention, conv, drift, linear, momentum, naive_lag, random_forest, recurrent, seq2seq, stack_encoder_ensemble_xgb, stack_rnn_arima_xgb, xgboost 

  00_naive_lag                     naive_lag                    <- (baseline — not an upstream notebook; required by spec R8)
  01_lstm                          recurrent                    <- deep-learning/1.lstm.ipynb
  02_bidirectional_lstm            recurrent                    <- deep-learning/2.bidirectional-lstm.ipynb
  03_lstm_2path                    recurrent                    <- deep-learning/3.lstm-2path.ipynb
  04_gru                           recurrent                    <- deep-learning/4.gru.ipynb
  05_bidirectional_gru             recurrent                    <- deep-learning/5.bidirectional-gru.ipynb
  06_gru_2path                     recurrent                    <- deep-learning/6.gru-2path.ipynb
  07_vanilla                       recurrent                    <- deep-learning/7.vanilla.ipynb
  08_b

## Run the whole catalogue

Equivalent to:

```bash
python -m stock_retrofit.cli evaluate --all --symbol KBANK
```

`NaiveLag` is inserted automatically whether or not you ask for it, and pinned to the top of the
table (spec R8).

In [7]:
from stock_retrofit.report import evaluate_symbol

models = evaluate_symbol("KBANK")
models

    00_naive_lag ... ok
    01_lstm ... ok
    02_bidirectional_lstm ... ok
    03_lstm_2path ... ok
    04_gru ... ok
    05_bidirectional_gru ... ok
    06_gru_2path ... ok
    07_vanilla ... ok
    08_bidirectional_vanilla ... ok
    09_vanilla_2path ... ok
    10_lstm_seq2seq ... ok
    11_bidirectional_lstm_seq2seq ... ok
    12_lstm_seq2seq_vae ... ok
    13_gru_seq2seq ... ok
    14_bidirectional_gru_seq2seq ... ok
    15_gru_seq2seq_vae ... ok
    16_attention_is_all_you_need ... 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was Tr

ok
    17_cnn_seq2seq ... ok
    18_dilated_cnn_seq2seq ... ok
    19_stack_rnn_arima_xgb ... ok
    20_stack_encoder_ensemble_xgb ... ok
    21_arima ... ok
    22_xgboost ... ok
KBANK — models
                        model symbol  folds   n   MASE beats_naive dir_acc coverage RMSE_ret sharpe_net sharpe_gross turnover
                 00_naive_lag  KBANK      8 480 1.0000          no       —       0%  0.01288      +0.00        +0.00     0.00
               13_gru_seq2seq  KBANK      8 480 1.0046          no   43.1%     100%  0.01282      +1.17        +1.92     0.13
       19_stack_rnn_arima_xgb  KBANK      8 480 1.0054          no   42.5%     100%  0.01286      +1.27        +1.28     0.00
                     21_arima  KBANK      8 480 1.0097          no   42.7%     100%  0.01288      -0.39        +1.48     0.40
          12_lstm_seq2seq_vae  KBANK      8 480 1.0097          no   41.0%     100%  0.01285      +0.95        +1.18     0.04
           15_gru_seq2seq_vae  KBANK      8 480 1

,model,symbol,upstream,folds,n,MASE,beats_naive,dir_acc,coverage,RMSE_ret,sharpe_net,sharpe_gross,turnover,status
0,00_naive_lag,KBANK,(baseline — not an upstream notebook; required...,8,480,1.000000,False,NaN,0.0,0.012883,0.000000,0.000000,0.000000,ok
1,13_gru_seq2seq,KBANK,deep-learning/13.gru-seq2seq.ipynb,8,480,1.004589,False,0.431250,1.0,0.012817,1.171671,1.922410,0.131250,ok
2,19_stack_rnn_arima_xgb,KBANK,stacking/stack-rnn-arima-xgb.ipynb,8,480,1.005428,False,0.425000,1.0,0.012860,1.270328,1.280466,0.002083,ok
3,21_arima,KBANK,stacking/stack-rnn-arima-xgb.ipynb (ARIMA comp...,8,480,1.009677,False,0.427083,1.0,0.012879,-0.388249,1.481133,0.402083,ok
4,12_lstm_seq2seq_vae,KBANK,deep-learning/12.lstm-seq2seq-vae.ipynb,8,480,1.009726,False,0.410417,1.0,0.012848,0.954150,1.177044,0.043750,ok
5,15_gru_seq2seq_vae,KBANK,deep-learning/15.gru-seq2seq-vae.ipynb,8,480,1.011641,False,0.414583,1.0,0.012848,1.502762,2.043500,0.077083,ok
6,14_bidirectional_gru_seq2seq,KBANK,deep-learning/14.bidirectional-gru-seq2seq.ipynb,8,480,1.012123,False,0.437500,1.0,0.012885,1.606277,2.482543,0.133333,ok
7,03_lstm_2path,KBANK,deep-learning/3.lstm-2path.ipynb,8,480,1.015802,False,0.410417,1.0,0.012932,0.512539,1.359585,0.137500,ok
8,02_bidirectional_lstm,KBANK,deep-learning/2.bidirectional-lstm.ipynb,8,480,1.016598,False,0.404167,1.0,0.012945,0.027361,1.032969,0.179167,ok
9,04_gru,KBANK,deep-learning/4.gru.ipynb,8,480,1.016932,False,0.454167,1.0,0.012892,2.095108,2.888594,0.118750,ok


## Reading the result

The `beats_naive` column is computed, not interpreted: **MASE < 1.00 beats the naive lag.**

Watch `dir_acc` too. These models sit around 40–45% directional accuracy — *below* a coin flip.
That is not a bug in the harness; it is what happens when a model trained to minimise squared
error on a near-random-walk return series is asked to call direction.

Note also the gap between `sharpe_gross` and `sharpe_net`. A model with a healthy frictionless
Sharpe and a negative net one has found a signal too small to pay for its own turnover — which is
the most common way a backtest lies.

In [8]:
from stock_retrofit.eval import summarise_beats

summary = summarise_beats(models)
print(f"{summary['beat_naive']} of {summary['ran']} models beat NaiveLag on MASE")
print("winners:", summary["winners"] or "none")

worst = models.loc[models["status"] == "ok"].nsmallest(5, "sharpe_net")[
    ["model", "MASE", "dir_acc", "sharpe_gross", "sharpe_net", "turnover"]]
print("\nlargest cost drag:")
worst

0 of 22 models beat NaiveLag on MASE
winners: none

largest cost drag:


,model,MASE,dir_acc,sharpe_gross,sharpe_net,turnover
22,22_xgboost,1.106399,0.408333,0.888803,-1.266115,0.337500
21,20_stack_encoder_ensemble_xgb,1.066729,0.406250,1.299231,-1.086316,0.397917
14,08_bidirectional_vanilla,1.022901,0.400000,0.925386,-1.018486,0.287500
20,07_vanilla,1.031820,0.406250,0.993074,-0.844701,0.320833
10,18_dilated_cnn_seq2seq,1.019616,0.425000,1.776116,-0.666456,0.350000
